# Lab 4: Motor Drivers and Open Loop Control
Data collection and testing notebook for dual motor drivers, calibration, and open-loop control.

In [ ]:
!pip install bleak colorama numpy matplotlib

In [ ]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import asyncio
import matplotlib.pyplot as plt

LOG.propagate = False

In [ ]:
ble = get_ble_controller()
ble.connect()

## Notification Handler
Handles IMU, ToF, stats, motor calibration responses, and test-done markers.

In [ ]:
imu_data = {
    'time': [], 'acc_x': [], 'acc_y': [], 'acc_z': [],
    'gyr_x': [], 'gyr_y': [], 'gyr_z': [],
    'mag_x': [], 'mag_y': [], 'mag_z': [],
}
tof1_data = {'time': [], 'dist': []}
tof2_data = {'time': [], 'dist': []}
tof_stats = {}
transfer_done = False
motor_response = None

def handle_data(uuid, message):
    global imu_data, tof1_data, tof2_data, tof_stats, transfer_done, motor_response
    msg = message.decode()

    if msg == "END":
        transfer_done = True
        return

    if msg == "TEST_DONE":
        motor_response = "done"
        return

    prefix = msg.split(":")[0] + ":"
    payload = msg[len(prefix):]
    parts = payload.split("|")

    if prefix == "I:":
        vals = [float(p) for p in parts]
        imu_data['time'].append(vals[0])
        for i, key in enumerate(['acc_x','acc_y','acc_z','gyr_x','gyr_y','gyr_z','mag_x','mag_y','mag_z']):
            imu_data[key].append(vals[i+1])
    elif prefix == "D1:":
        tof1_data['time'].append(float(parts[0]))
        tof1_data['dist'].append(float(parts[1]))
    elif prefix == "D2:":
        tof2_data['time'].append(float(parts[0]))
        tof2_data['dist'].append(float(parts[1]))
    elif prefix == "S1:":
        tof_stats['s1'] = {'n': int(parts[0]), 'mean': float(parts[1]), 'std': float(parts[2])}
    elif prefix == "S2:":
        tof_stats['s2'] = {'n': int(parts[0]), 'mean': float(parts[1]), 'std': float(parts[2])}
    elif prefix == "CAL:":
        motor_response = f"cal={parts[0]}"

try:
    ble.stop_notify(ble.uuid["RX_STRING"])
except Exception:
    pass

ble.start_notify(ble.uuid["RX_STRING"], handle_data)
print("Notification handler ready")

---
## 1. Basic Motor Test
Test each motor individually. **Place the car on its side** so wheels spin freely.

Format: `MOTOR_CMD` takes `left_pwm|right_pwm|timeout_ms`
- PWM range: -255 to 255 (negative = reverse)
- Timeout: auto-stop after N ms (safety feature)

In [ ]:
# Test left motor forward (2 seconds)
ble.send_command(CMD.MOTOR_CMD, "150|0|2000")
print("Left motor forward...")
time.sleep(2.5)
print("Done")

In [ ]:
# Test right motor forward (2 seconds)
ble.send_command(CMD.MOTOR_CMD, "0|150|2000")
print("Right motor forward...")
time.sleep(2.5)
print("Done")

In [ ]:
# Test both motors forward (2 seconds)
ble.send_command(CMD.MOTOR_CMD, "150|150|2000")
print("Both motors forward...")
time.sleep(2.5)
print("Done")

In [ ]:
# Test both motors reverse (2 seconds)
ble.send_command(CMD.MOTOR_CMD, "-150|-150|2000")
print("Both motors reverse...")
time.sleep(2.5)
print("Done")

In [ ]:
# Emergency stop
ble.send_command(CMD.MOTOR_STOP, "")
print("Motors stopped")

---
## 2. Minimum PWM Threshold
Find the lowest PWM value that overcomes static friction and gets the robot moving.
Run on the ground (not on its side).

In [ ]:
# Sweep PWM values to find minimum for forward movement
pwm_values = list(range(30, 200, 10))
results = []

for pwm in pwm_values:
    print(f"Testing PWM = {pwm}... ", end="")
    ble.send_command(CMD.MOTOR_CMD, f"{pwm}|{pwm}|1500")
    time.sleep(2)
    ble.send_command(CMD.MOTOR_STOP, "")
    
    moved = input("Did it move? (y/n): ").strip().lower()
    results.append({'pwm': pwm, 'moved': moved == 'y'})
    time.sleep(1)

print("\nResults:")
for r in results:
    status = 'MOVED' if r['moved'] else 'no movement'
    print(f"  PWM {r['pwm']:>3}: {status}")

In [ ]:
# Same sweep for on-axis turns
turn_results = []

for pwm in pwm_values:
    print(f"Testing turn PWM = {pwm}... ", end="")
    ble.send_command(CMD.MOTOR_CMD, f"{-pwm}|{pwm}|1500")
    time.sleep(2)
    ble.send_command(CMD.MOTOR_STOP, "")
    
    moved = input("Did it turn? (y/n): ").strip().lower()
    turn_results.append({'pwm': pwm, 'moved': moved == 'y'})
    time.sleep(1)

print("\nTurn Results:")
for r in turn_results:
    status = 'TURNED' if r['moved'] else 'no movement'
    print(f"  PWM {r['pwm']:>3}: {status}")

---
## 3. Motor Calibration
Adjust the calibration factor so both motors run at the same speed.
The robot should drive in a straight line for 2m / 6ft.

In [ ]:
# Set calibration factor (right_motor_pwm *= cal)
# If robot veers right, increase cal (right motor needs more power)
# If robot veers left, decrease cal
cal_factor = 1.0

ble.send_command(CMD.SET_MOTOR_CAL, f"{cal_factor}")
time.sleep(0.5)
print(f"Calibration set to {cal_factor}")

In [ ]:
# Straight line test: drive forward for 3 seconds at moderate speed
test_pwm = 120
ble.send_command(CMD.MOTOR_CMD, f"{test_pwm}|{test_pwm}|3000")
print(f"Driving straight at PWM={test_pwm} for 3s...")
time.sleep(3.5)
print("Done - measure deviation from straight line")

---
## 4. Open Loop Control Demo
Run a pre-programmed sequence: forward, left turn, right turn.
Or compose your own sequence below.

In [ ]:
# Run the built-in test sequence on the Arduino
motor_response = None
ble.send_command(CMD.MOTOR_TEST_SEQ, "150")
print("Running test sequence...")

while motor_response != "done":
    await asyncio.sleep(0.1)

print("Test sequence complete!")

In [ ]:
# Custom open-loop sequence from Python
# Define sequence as list of (left_pwm, right_pwm, duration_ms)
sequence = [
    (150, 150, 1500),    # Forward 1.5s
    (0, 0, 300),         # Pause
    (-120, 120, 800),    # Turn left ~90 deg
    (0, 0, 300),         # Pause
    (150, 150, 1500),    # Forward 1.5s
    (0, 0, 300),         # Pause
    (120, -120, 800),    # Turn right ~90 deg
    (0, 0, 300),         # Pause
    (150, 150, 1000),    # Forward 1s
]

print("Running custom sequence...")
for left, right, dur in sequence:
    if left == 0 and right == 0:
        ble.send_command(CMD.MOTOR_STOP, "")
    else:
        ble.send_command(CMD.MOTOR_CMD, f"{left}|{right}|{dur}")
    time.sleep(dur / 1000.0 + 0.05)

ble.send_command(CMD.MOTOR_STOP, "")
print("Sequence complete!")

---
## 5. PWM Frequency Analysis (5000-level)
Discuss whether the default `analogWrite` frequency is adequate.
The Artemis Apollo3 default PWM frequency depends on the timer configuration.

In [ ]:
# Measure effective motor response at different PWM duty cycles
# Run each for a short time and observe/record behavior
test_pwms = [50, 80, 100, 120, 150, 180, 200, 255]

for pwm in test_pwms:
    print(f"PWM = {pwm}...")
    ble.send_command(CMD.MOTOR_CMD, f"{pwm}|{pwm}|2000")
    time.sleep(2.5)
    ble.send_command(CMD.MOTOR_STOP, "")
    time.sleep(1)

print("Done")

---
## Disconnect

In [ ]:
# Safety: always stop motors before disconnecting
ble.send_command(CMD.MOTOR_STOP, "")
# ble.disconnect()